---

Set up the notebook.

---

In [82]:
import os
import re
import json
import shutil
import logging
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import yaml

from PIL import Image
import cv2 as cv

from tqdm import tqdm
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parents[1]

RAW_DIR      = PROJECT_ROOT / "datasets" / "raw"
INTERIM_DIR  = PROJECT_ROOT / "datasets" / "interim"
REPORT_DIR   = PROJECT_ROOT / "datasets" / "reports"
CONFIG_DIR   = PROJECT_ROOT / "config"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = REPORT_DIR / f"taxonomy_mapping_log_{timestamp}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

TAXONOMY = ["person", "stairs", "door", "chair", "table", "pole", "bicycle", "vehicle"]

logger.info(f"RAW_DIR: {RAW_DIR}")
logger.info(f"INTERIM_DIR: {INTERIM_DIR}")
logger.info(f"REPORT_DIR: {REPORT_DIR}")
logger.info(f"CONFIG_DIR: {CONFIG_DIR}")
logger.info(f"TAXONOMY: {TAXONOMY}")

2026-08-09 00:10:50,077 [INFO] RAW_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw
2026-08-09 00:10:50,079 [INFO] INTERIM_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim
2026-08-09 00:10:50,080 [INFO] REPORT_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports
2026-08-09 00:10:50,081 [INFO] CONFIG_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\config
2026-08-09 00:10:50,082 [INFO] TAXONOMY: ['person', 'stairs', 'door', 'chair', 'table', 'pole', 'bicycle', 'vehicle']


---

Create the mapping rules for the 8 taxonomy

---

In [83]:
excluded_pairs = {("roboflow", "indoor_objects_roboflow")}

MAPPING_RULES = {
    ("kaggle", "indoor_object_detection"): {
        "door": "door", "cabinetDoor": "door", "refrigeratorDoor": "door", "openedDoor": "door",
        "chair": "chair", "table": "table", "pole": "pole",
        "window": None, "cabinet": None, "couch": None,
    },
    ("kaggle", "obstacle_detection_kaggle"): {
        "Person": "person", "Stairs": "stairs", "Electrical Pole": "pole",
        "Bike": "bicycle", "Chair": "chair",
        "Car": "vehicle", "Truck": "vehicle", "Bus": "vehicle", "Motorcycle": "vehicle",
        "Building": None, "Traffic sign": None, "Road": None, "Dustbin": None, "Dog": None,
        "Manhole": None, "Tree": None, "Guard rail": None, "Pedestrian crosswalk": None,
        "Bench": None, "Traffic Cone": None, "Fire hydrant": None, "Teraffic Barrel": None,
        "Plant Pot": None, "Electrical Box": None, "Bicycle Rack": None,
    },
    ("roboflow", "footpath_detection"): {
        "person_back": "person", "person_front": "person", "vehicle": "vehicle",
        "obstacle_1": None, "obstacle_2": None,
    },
    ("roboflow", "indoor_detection_vineeth"): {
        "cabinetDoor": "door", "door": "door", "openedDoor": "door", "refrigeratorDoor": "door",
        "chair": "chair", "table": "table", "pole": "pole",
        "cabinet": None, "couch": None, "window": None,
    },
    ("roboflow", "indoor_objects_5iwhq"): {
        "chair": "chair", "door": "door", "door-frame": "door", "stairs": "stairs", "table": "table",
        "bed": None, "shower": None, "sink": None, "sofa": None, "toilet": None,
    },
    ("roboflow", "obstacle_detection_roboflow"): {
        "Bicycle": "bicycle", "Car": "vehicle", "Bus": "vehicle", "Motorcycle": "vehicle",
        "Person": "person", "Electric pole": "pole",
        "Dog": None, "Traffic signs": None, "Tree": None, "Uncovered manhole": None,
    },
    ("roboflow", "outdoor_objects"): {
        "person": "person", "street light": "pole",
        "building": None, "foot path": None, "plants": None, "road": None, "trees": None,
    },
    ("roboflow", "pedestrian_walk"): {
        "B": "person", "F": "person", "M": "person",
    },
    ("roboflow", "stairs_detection"): {
        "stairs": "stairs",
    },
}

mapping_records = []
unresolved = []

for _, row in class_mapping_df.iterrows():
    key = (row["source"], row["dataset"])
    if key in excluded_pairs:
        continue

    rules = MAPPING_RULES.get(key)
    if rules is None:
        unresolved.append((row["source"], row["dataset"], row["class_id"], row["class_name"], "no ruleset defined"))
        continue

    target = rules.get(row["class_name"], "__NOT_IN_RULESET__")
    if target == "__NOT_IN_RULESET__":
        unresolved.append((row["source"], row["dataset"], row["class_id"], row["class_name"], "class not in ruleset"))
        continue

    mapping_records.append({
        "source": row["source"],
        "dataset": row["dataset"],
        "source_class_id": int(row["class_id"]),
        "source_class": row["class_name"],
        "target_class": target,
        "status": "mapped" if target else "unmapped",
        "reason": None if target else "outside_project_taxonomy",
    })

print(f"Mapped/unmapped records built: {len(mapping_records)}")
print(f"Unresolved (ruleset gap — needs fixing): {len(unresolved)}")
for u in unresolved:
    print(f"  {u}")

Mapped/unmapped records built: 81
Unresolved (ruleset gap — needs fixing): 0


---

Checks some special datasets manually

---

In [84]:
mapping_records.append({
    "source": "kaggle",
    "dataset": "light_poles",
    "source_class_id": 0,
    "source_class": "light pole",
    "target_class": "pole",
    "status": "mapped",
    "reason": None,
})

print("light_poles added.")
print(f"Total mapping records now: {len(mapping_records)}")

light_poles added.
Total mapping records now: 82


In [87]:
mapping_records.append({
    "source": "kaggle",
    "dataset": "pedestrian_detection",
    "source_class_id": 0,
    "source_class": "person",
    "target_class": "person",
    "status": "mapped",
    "reason": None,
})

print("pedestrian_detection added.")
print(f"Total mapping records now: {len(mapping_records)}")

pedestrian_detection added.
Total mapping records now: 83


In [96]:
mapping_records.append({
    "source": "roboflow",
    "dataset": "footpath_detection",
    "source_class_id": 6,
    "source_class": "bicycle (undeclared in yaml, visually confirmed)",
    "target_class": "bicycle",
    "status": "mapped",
    "reason": None,
})

print("footpath_detection class 6 added.")

footpath_detection class 6 added.


In [103]:
mapping_records.append({
    "source": "roboflow",
    "dataset": "footpath_detection",
    "source_class_id": 7,
    "source_class": "vehicle/car (undeclared in yaml, visually confirmed)",
    "target_class": "vehicle",
    "status": "mapped",
    "reason": None,
})

print("footpath_detection class 7 added.")
print(f"Total mapping records now: {len(mapping_records)}")

footpath_detection class 7 added.
Total mapping records now: 93


In [104]:
for class_id, class_name in enumerate(["crosswalk", "speedlimit", "stop", "trafficlight"]):
    mapping_records.append({
        "source": "kaggle",
        "dataset": "road_sign_detection",
        "source_class_id": class_id,
        "source_class": class_name,
        "target_class": None,
        "status": "unmapped",
        "reason": "outside_project_taxonomy",
    })

print("road_sign_detection added (all unmapped).")
print(f"Total mapping records now: {len(mapping_records)}")

road_sign_detection added (all unmapped).
Total mapping records now: 97


In [106]:
mapping_records.append({
    "source": "roboflow",
    "dataset": "outdoor_objects",
    "source_class_id": 7,
    "source_class": "car/vehicle (undeclared in yaml, visually confirmed)",
    "target_class": "vehicle",
    "status": "mapped",
    "reason": None,
})

print("outdoor_objects class 7 added.")
print(f"Total mapping records now: {len(mapping_records)}")

outdoor_objects class 7 added.
Total mapping records now: 98


In [108]:
mapping_df = pd.DataFrame(mapping_records)

validation_issues = []

for (source, dataset), group in mapping_df.groupby(["source", "dataset"]):
    if dataset == "road_sign_detection":
        continue

    dataset_dir = INTERIM_DIR / source / dataset
    label_files = [f for f in dataset_dir.rglob("*.txt") if f.stat().st_size > 0]

    actual_ids_used = set()
    for lf in label_files:
        for line in lf.read_text().strip().splitlines():
            parts = line.split()
            if parts:
                actual_ids_used.add(int(parts[0]))

    mapped_ids = set(group["source_class_id"])
    missing_from_mapping = actual_ids_used - mapped_ids

    if missing_from_mapping:
        validation_issues.append({
            "source": source, "dataset": dataset,
            "issue": "class_id used in data but not in mapping",
            "ids": missing_from_mapping,
        })

print(f"Remaining validation issues: {len(validation_issues)}")
for issue in validation_issues:
    print(f"  {issue['source']}/{issue['dataset']}: {issue['issue']} — {issue['ids']}")

Remaining validation issues: 0


---

Create taxonomy_mapping.yaml

---

In [109]:
mapping_output_path = CONFIG_DIR / "taxonomy_mapping.yaml"

mapping_output = {
    "taxonomy": TAXONOMY,
    "excluded_datasets": [{"source": s, "dataset": d} for s, d in excluded_pairs],
    "mappings": mapping_records,
}

with open(mapping_output_path, "w") as f:
    yaml.dump(mapping_output, f, sort_keys=False, default_flow_style=False)

print(f"Saved {len(mapping_records)} class mappings to {mapping_output_path}")
print(f"Mapped: {sum(1 for r in mapping_records if r['status'] == 'mapped')}")
print(f"Unmapped: {sum(1 for r in mapping_records if r['status'] == 'unmapped')}")

Saved 98 class mappings to D:\SIT374\WalkBuddy-T2-2026\ML_side\config\taxonomy_mapping.yaml
Mapped: 48
Unmapped: 50


In [110]:
gap_check = []

for _, row in inventory_df.iterrows():
    if (row["source"], row["dataset"]) in excluded_pairs:
        continue
    if row["label_format"] != "txt":
        continue

    dataset_dir = INTERIM_DIR / row["source"] / row["dataset"]
    all_files = [f for f in dataset_dir.rglob("*") if f.is_file()]

    image_files = [f for f in all_files if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    label_files = {f.stem: f for f in all_files if f.suffix.lower() == ".txt"}

    for img in image_files:
        lf = label_files.get(img.stem)
        if lf is None or lf.stat().st_size == 0:
            gap_check.append({
                "source": row["source"], "dataset": row["dataset"],
                "stem": img.stem, "path": str(img.relative_to(INTERIM_DIR)),
            })

gap_df = pd.DataFrame(gap_check)
gap_df["flagged_junk"] = gap_df["stem"].apply(is_junk_filename)
real_gap = gap_df[~gap_df["flagged_junk"]]

print(f"Total gap (no_file + empty_file): {len(gap_df)}")
print(f"Real gap after junk filter: {len(real_gap)}")
print(real_gap.groupby("dataset").size())

Total gap (no_file + empty_file): 757
Real gap after junk filter: 736
dataset
footpath_detection                         8
indoor_detection_vineeth                   1
indoor_object_detection                  103
indoor_objects_5iwhq                       2
obstacle_detection_kaggle                365
obstacle_detection_roboflow              161
outdoor_objects                           36
revised_pedestrian_obstacle_detection     58
stairs_detection                           2
dtype: int64


In [114]:
def remap_dataset(source, dataset, id_map):
    dataset_dir = INTERIM_DIR / source / dataset
    label_files = [f for f in dataset_dir.rglob("*.txt") if f.stat().st_size > 0]

    remapped_count = 0
    dropped_lines = 0

    for lf in label_files:
        lines = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            old_id = int(parts[0])
            new_id = id_map.get(old_id)
            if new_id is None:
                dropped_lines += 1
                continue
            new_lines.append(" ".join([str(new_id)] + parts[1:]))

        lf.write_text("\n".join(new_lines))
        remapped_count += 1

    return remapped_count, dropped_lines

taxonomy_index = {name: i for i, name in enumerate(TAXONOMY)}
mapping_df = pd.DataFrame(mapping_records)

remap_summary = []

for (source, dataset), group in mapping_df.groupby(["source", "dataset"]):
    if dataset == "road_sign_detection":
        continue

    id_map = {}
    for _, r in group.iterrows():
        id_map[r["source_class_id"]] = taxonomy_index[r["target_class"]] if r["status"] == "mapped" else None

    files_touched, dropped = remap_dataset(source, dataset, id_map)
    remap_summary.append({"source": source, "dataset": dataset, "files_touched": files_touched, "lines_dropped": dropped})

remap_df = pd.DataFrame(remap_summary)
print(remap_df.to_string(index=False))

  source                     dataset  files_touched  lines_dropped
  kaggle     indoor_object_detection           1219            938
  kaggle                 light_poles            909              0
  kaggle   obstacle_detection_kaggle          23916          17483
  kaggle        pedestrian_detection           2224              0
roboflow          footpath_detection           1687           2261
roboflow    indoor_detection_vineeth           2889           2207
roboflow        indoor_objects_5iwhq           1487           1104
roboflow obstacle_detection_roboflow           9074           7294
roboflow             outdoor_objects           2370           5815
roboflow             pedestrian_walk           1000              0
roboflow            stairs_detection            231              0


In [115]:
for _, row in remap_df.iterrows():
    dataset_dir = INTERIM_DIR / row["source"] / row["dataset"]
    remaining_content = sum(
        1 for f in dataset_dir.rglob("*.txt") if f.stat().st_size > 0
    )
    print(f"{row['dataset']}: {remaining_content} label files still have content after remap")

indoor_object_detection: 1184 label files still have content after remap
light_poles: 909 label files still have content after remap
obstacle_detection_kaggle: 11253 label files still have content after remap
pedestrian_detection: 2224 label files still have content after remap
footpath_detection: 1616 label files still have content after remap
indoor_detection_vineeth: 2822 label files still have content after remap
indoor_objects_5iwhq: 1046 label files still have content after remap
obstacle_detection_roboflow: 5683 label files still have content after remap
outdoor_objects: 1366 label files still have content after remap
pedestrian_walk: 1000 label files still have content after remap
stairs_detection: 231 label files still have content after remap


---

Copy mapped data into a new folder

---

In [128]:
INTERIM_MAPPED_DIR = PROJECT_ROOT / "datasets" / "interim_mapped"

if INTERIM_MAPPED_DIR.exists():
    shutil.rmtree(INTERIM_MAPPED_DIR)
INTERIM_MAPPED_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
skipped_no_image = 0

for source_dir in [d for d in INTERIM_DIR.iterdir() if d.is_dir()]:
    for dataset_dir in [d for d in source_dir.iterdir() if d.is_dir()]:
        if (source_dir.name, dataset_dir.name) in excluded_pairs:
            continue

        all_images = {f.stem: f for f in dataset_dir.rglob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png"}}
        label_files = [f for f in dataset_dir.rglob("*.txt") if f.stat().st_size > 0]

        for lf in label_files:
            img_path = all_images.get(lf.stem)

            if img_path is None:
                skipped_no_image += 1
                continue

            rel_path = img_path.relative_to(INTERIM_DIR)
            dest_img = INTERIM_MAPPED_DIR / rel_path
            dest_label = dest_img.with_suffix(".txt")

            dest_img.parent.mkdir(parents=True, exist_ok=True)

            shutil.copy2(img_path, dest_img)
            shutil.copy2(lf, dest_label)
            copied += 1

print(f"Copied {copied} image+label pairs into {INTERIM_MAPPED_DIR}")
print(f"Skipped (genuinely no matching image found): {skipped_no_image}")

Copied 35633 image+label pairs into D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim_mapped
Skipped (genuinely no matching image found): 0


---

Class count for each taxonomy

---

In [129]:
class_counts = Counter()
for f in INTERIM_MAPPED_DIR.rglob("*.txt"):
    if f.stat().st_size == 0:
        continue
    for line in f.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            class_id = int(parts[0])
            class_counts[TAXONOMY[class_id]] += 1
for cls in TAXONOMY:
    print(f"{cls}: {class_counts.get(cls, 0)}")

person: 25714
stairs: 3414
door: 21839
chair: 5072
table: 8608
pole: 5665
bicycle: 5307
vehicle: 16407


---

Comprise a report

---

In [130]:
print("=" * 60)
print("PIPELINE SUMMARY — pre-scaling checkpoint")
print("=" * 60)

print(f"\nTaxonomy ({len(TAXONOMY)} classes): {TAXONOMY}")

print(f"\nExcluded datasets: {len(excluded_pairs)}")
for s, d in excluded_pairs:
    print(f"  {s}/{d}")

print(f"\nTaxonomy mapping: {len(mapping_records)} source classes recorded")
print(f"  Mapped: {sum(1 for r in mapping_records if r['status'] == 'mapped')}")
print(f"  Unmapped: {sum(1 for r in mapping_records if r['status'] == 'unmapped')}")
print(f"  Saved to: {CONFIG_DIR / 'taxonomy_mapping.yaml'}")

print(f"\nRemap results (files touched / lines dropped):")
print(remap_df.to_string(index=False))

print(f"\nAuto-annotation gap fill (Track A, conf=0.15):")
print(f"  Processed: {len(results_df)}")
print(f"  Now labeled: {(results_df['num_detections'] > 0).sum()}")
print(f"  Still empty (left as-is, skipped): {(results_df['num_detections'] == 0).sum()}")

print(f"\ninterim_mapped/ folder (remapped, content-only copy):")
mapped_images = list(INTERIM_MAPPED_DIR.rglob("*.jpg")) + list(INTERIM_MAPPED_DIR.rglob("*.jpeg")) + list(INTERIM_MAPPED_DIR.rglob("*.png"))
mapped_labels = [f for f in INTERIM_MAPPED_DIR.rglob("*.txt")]
print(f"  Images: {len(mapped_images)}")
print(f"  Labels: {len(mapped_labels)}")

print(f"\nPer-class instance counts (interim_mapped/):")
class_counts = Counter()
for f in mapped_labels:
    if f.stat().st_size == 0:
        continue
    for line in f.read_text().strip().splitlines():
        parts = line.split()
        if parts:
            class_counts[TAXONOMY[int(parts[0])]] += 1
for cls in TAXONOMY:
    print(f"  {cls}: {class_counts.get(cls, 0)}")

print(f"\nFlagged for later review (high drop-count datasets, mapping confirmed correct):")
print("  kaggle/indoor_object_detection, kaggle/obstacle_detection_kaggle, roboflow/obstacle_detection_roboflow")

print(f"\nHeld back entirely (no taxonomy overlap): kaggle/road_sign_detection")

print(f"\nStill open / not yet done:")
print("  - Track B (custom stairs/door/pole detector) — not built")
print("  - 619 images from Track A gap-fill still unlabeled — skipped for now, not deleted")
print("  - Scaling (resize + bbox rescale) — NEXT")

print("\n" + "=" * 60)

PIPELINE SUMMARY — pre-scaling checkpoint

Taxonomy (8 classes): ['person', 'stairs', 'door', 'chair', 'table', 'pole', 'bicycle', 'vehicle']

Excluded datasets: 1
  roboflow/indoor_objects_roboflow

Taxonomy mapping: 98 source classes recorded
  Mapped: 48
  Unmapped: 50
  Saved to: D:\SIT374\WalkBuddy-T2-2026\ML_side\config\taxonomy_mapping.yaml

Remap results (files touched / lines dropped):
  source                     dataset  files_touched  lines_dropped
  kaggle     indoor_object_detection           1219            938
  kaggle                 light_poles            909              0
  kaggle   obstacle_detection_kaggle          23916          17483
  kaggle        pedestrian_detection           2224              0
roboflow          footpath_detection           1687           2261
roboflow    indoor_detection_vineeth           2889           2207
roboflow        indoor_objects_5iwhq           1487           1104
roboflow obstacle_detection_roboflow           9074           7294

---

End of annotation cleaning process (my suffering can finally ends)

---